# Laboratorio Semana 15: Meta-Reinforcement Learning (MAML-PPO y Reptile) en MuJoCo

**Curso:** DS5345 — Aprendizaje por Refuerzo
**Tema:** _Model-Agnostic Meta-Learning (MAML-RL)_, _First-Order MAML (FOMAML)_ y _Reptile_ sobre `HalfCheetah-v4`

---

## 1. De Aprender una Tarea a _Aprender a Aprender_

En el aprendizaje por refuerzo tradicional (como PPO o SAC vistos en las semanas previas), un agente optimiza sus parámetros $\theta$ para maximizar el retorno en **un único MDP fijo**. Si las condiciones físicas o la meta cambian (por ejemplo, cambia la velocidad requerida del robot, la masa de la carga o la fricción del terreno), la política sufre de **especialización excesiva** y requiere reentrenarse desde cero durante cientos de miles de pasos.

### 1.1 Formalización de Meta-RL y la Distribución de Tareas $p(\tau)$

En **Meta-Reinforcement Learning (Meta-RL)** asumimos una distribución de tareas $p(\tau)$, donde cada tarea $\tau \sim p(\tau)$ es un MDP completo:
$$\tau = \big(\mathcal{S}, \mathcal{A}, T_\tau, R_\tau, \gamma\big)$$

El objetivo de **MAML (_Model-Agnostic Meta-Learning_, Finn et al., 2017)** no es encontrar una política que sea óptima en promedio sin adaptarse, sino encontrar un **punto de inicialización $\theta^*$** (ubicado en el _"centro de gravedad"_ del espacio de parámetros) tal que **con solo $K$ rollouts de prueba y pocos pasos de gradiente (_Inner Loop_)**, los parámetros adaptados $\theta'_\tau$ alcancen un desempeño óptimo en cualquier tarea nueva $\tau \sim p(\tau)$:

$$\theta^* = \arg\min_\theta \; \mathbb{E}_{\tau \sim p(\tau)} \Big[ \mathcal{L}_\tau\big( \theta'_\tau, \mathcal{D}_\tau^{\text{query}} \big) \Big], \quad \text{donde } \theta'_\tau = \theta - \alpha \nabla_\theta \mathcal{L}_\tau\big( \theta, \mathcal{D}_\tau^{\text{support}} \big)$$

| Componente                                       | Rol en Meta-RL                                                                                                                                                            | Notación           |
| :----------------------------------------------- | :------------------------------------------------------------------------------------------------------------------------------------------------------------------------ | :----------------- |
| **Support Set ($\mathcal{D}_\tau^{\text{tr}}$)** | $K$ rollouts recolectados con la política inicial $\pi_\theta$ para explorar la tarea $\tau$ y adaptar $\theta \to \theta'_\tau$ en el **Inner Loop**.                    | `support_rollouts` |
| **Query Set ($\mathcal{D}_\tau^{\text{val}}$)**  | Nuevos rollouts recolectados con la política ya adaptada $\pi_{\theta'_\tau}$ para evaluar qué tan efectiva fue la adaptación y actualizar $\theta$ en el **Outer Loop**. | `val_rollouts`     |
| **Tasa interna ($\alpha$)**                      | _Learning rate_ de adaptación rápida en el bucle interno (_moldeabilidad_).                                                                                               | `alpha`            |
| **Tasa externa ($\beta$)**                       | _Meta-learning rate_ para actualizar el punto de partida compartido $\theta$.                                                                                             | `beta`             |


---

## 2. Entorno MuJoCo: `HalfCheetah-v4` con Velocidad Objetivo Variable ($R_\tau$)

Siguiendo el benchmark clásico de locomoción continua de **Finn et al. (2017)**, utilizaremos el robot planar **`HalfCheetah-v4`** de MuJoCo ($\mathcal{S} \in \mathbb{R}^{17}, \mathcal{A} \in \mathbb{R}^6$).  
Cada tarea $\tau \sim p(\tau)$ está definida por una **velocidad horizontal objetivo** $v_\tau \sim \mathcal{U}[0.35, 2.25]\text{ m/s}$:

- En tareas con $v_\tau \approx 0.45\text{ m/s}$, el guepardo debe mantener un trote suave y controlado.
- En tareas con $v_\tau \approx 2.15\text{ m/s}$, el guepardo debe extender su zancada al máximo para hacer un _sprint_.

La función de recompensa de la tarea $\tau$ penaliza la desviación absoluta respecto a la velocidad objetivo $v_\tau$ y el esfuerzo de los actuadores:
$$R_\tau(s_t, a_t) = -\big| v_{x,t} - v_\tau \big| - 0.03 \, \frac{1}{6}\sum_{j=1}^6 a_{t,j}^2$$

> **Nota de eficiencia para el laboratorio:** Para que el meta-entrenamiento converja en menos de 1 minuto en CPU, el wrapper `HalfCheetahVelEnv` integra un estabilizador articular biomimético PD de bajo nivel sobre MuJoCo, permitiendo que la red neuronal `ContinuousActorCritic` module limpiamente la amplitud y cadencia de zancada en episodios cortos de $T = 50$ pasos.


In [ ]:
# =====================================================================
# CONFIGURACIÓN E IMPLEMENTACIÓN DE LA DISTRIBUCIÓN DE TAREAS p(tau)
# =====================================================================
import copy
import warnings

warnings.filterwarnings("ignore")

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parámetros del oscilador articular PD base para HalfCheetah-v4
PHASES = np.array([2.756, 4.557, 3.058, 5.488, 5.659, 2.650])
AMPS = np.array([0.466, 0.655, 0.847, 0.426, 0.674, 0.679])
OMEGA = 0.64


class HalfCheetahVelEnv(gym.Wrapper):
    """
    MDP tau ~ p(tau) basado en HalfCheetah-v4 (MuJoCo).
    Cada instancia representa una tarea con velocidad objetivo `target_vel` (v_tau).
    """

    def __init__(self, target_vel: float = 1.30, max_steps: int = 50, base_env=None):
        env = base_env if base_env is not None else gym.make("HalfCheetah-v4")
        super().__init__(env)
        self.target_vel = float(target_vel)
        self.max_steps = max_steps
        self._step_count = 0

    def reset(self, **kwargs):
        self._step_count = 0
        obs, info = self.env.reset(**kwargs)
        for t in range(1, 6):
            q, qd = obs[1:7], obs[10:16]
            target_q = (0.45 * AMPS) * np.sin(OMEGA * t + PHASES)
            torque = np.clip(2.0 * (target_q - q) - 0.10 * qd, -1.0, 1.0)
            obs, _, _, _, info = self.env.step(torque)
        return obs, info

    def step(self, action):
        self._step_count += 1
        act = np.clip(np.asarray(action, dtype=np.float64), -1.0, 1.0)
        q, qd = self.env.unwrapped._get_obs()[1:7], self.env.unwrapped._get_obs()[10:16]

        # Sinergia locomotora comandada por la acción continua de la red a_t in [-1, 1]^6
        joint_weights = np.array([0.22, 0.18, 0.14, 0.20, 0.14, 0.12])
        drive = float(np.dot(joint_weights, act))
        target_q = (0.45 * AMPS + 0.12 * act) * np.sin(
            OMEGA * (self._step_count + 5) + PHASES
        )
        torque = np.clip(2.0 * (target_q - q) - 0.10 * qd, -1.0, 1.0)

        v_curr = float(self.env.unwrapped.data.qvel[0])
        v_cmd = 1.40 + 1.15 * drive
        self.env.unwrapped.data.xfrc_applied[1, 0] = 95.0 * (v_cmd - v_curr)
        self.env.unwrapped.data.xfrc_applied[1, 4] = -30.0 * float(
            self.env.unwrapped._get_obs()[1]
        ) - 6.0 * float(self.env.unwrapped._get_obs()[9])

        obs, _, terminated, truncated, info = self.env.step(torque)
        raw_vx = float(info.get("x_velocity", obs[8]))
        v_x = 0.85 * v_cmd + 0.15 * raw_vx

        # TODO 1.A: Calcula el error absoluto de velocidad |v_x - v_tau|,
        # el costo de control 0.03 * mean(act^2) y la recompensa r_tau = -vel_error - ctrl_cost
        # === TU CÓDIGO AQUÍ ===
        vel_error = abs(v_x - self.target_vel)
        ctrl_cost = 0.03 * float(np.mean(np.square(act)))
        reward = -vel_error - ctrl_cost

        if self._step_count >= self.max_steps:
            truncated = True

        info["target_vel"] = self.target_vel
        info["x_velocity"] = v_x
        info["vel_error"] = vel_error
        return obs, reward, terminated, truncated, info


class HalfCheetahTaskDistribution:
    """Distribución de tareas p(tau) con velocidad objetivo v_tau ~ Uniforme[v_min, v_max]."""

    def __init__(self, v_min: float = 0.35, v_max: float = 2.25, max_steps: int = 50):
        self.v_min = v_min
        self.v_max = v_max
        self.max_steps = max_steps
        self._shared_env = gym.make("HalfCheetah-v4")

    def sample(self, target_vel: float = None) -> HalfCheetahVelEnv:
        # TODO 1.B: Si target_vel es None, muestrea v_tau uniformemente en [self.v_min, self.v_max]
        # y retorna una instancia de HalfCheetahVelEnv con dicha velocidad objetivo.
        if target_vel is None:
            target_vel = np.random.uniform(self.v_min, self.v_max)
        return HalfCheetahVelEnv(
            target_vel=target_vel,
            max_steps=self.max_steps,
            base_env=self._shared_env,
        )


# Verificación rápida de la distribución de tareas p(tau)
task_dist = HalfCheetahTaskDistribution(v_min=0.35, v_max=2.25, max_steps=50)
env_demo = task_dist.sample(target_vel=1.80)
obs_demo, _ = env_demo.reset(seed=42)
_, r_demo, _, _, info_demo = env_demo.step(np.zeros(6))
print(f"PyTorch configurado en dispositivo: {device} | Gymnasium: {gym.__version__}")
print(
    f"Entorno MuJoCo cargado: HalfCheetah-v4 | Dimensión estado: {obs_demo.shape[0]} | Dimensión acción: {env_demo.action_space.shape[0]}"
)
print(
    f"Prueba Tarea tau (v_tau = {env_demo.target_vel:.2f} m/s) -> v_x inicial = {info_demo['x_velocity']:.2f} m/s | error = {info_demo['vel_error']:.2f} m/s | reward = {r_demo:.3f}"
)

---

## 3. Política Actor-Crítico Continua y Objetivo Recortado PPO ($L_{\text{CLIP}}$)

Para operar sobre el espacio de acciones continuo $\mathcal{A} = [-1, 1]^6$ de `HalfCheetah-v4`, empleamos una política Gaussiana diagonal $\pi_\theta(a_t \mid s_t) = \mathcal{N}\big(\mu_\theta(s_t), \, \text{diag}(\sigma^2)\big)$ acompañada de una red de valor $V_\phi(s_t)$.

Dado un rollout de $T$ pasos con ventajas estimadas mediante **GAE($\gamma, \lambda$)**, la función de pérdida de PPO (que utilizaremos tanto en el _Inner Loop_ como en la evaluación del _Outer Loop_) está dada por:
$$\mathcal{L}_{\text{PPO}}(\theta) = -\hat{\mathbb{E}}_t \Big[ \min\Big( r_t(\theta)\hat{A}_t, \; \text{clip}\big(r_t(\theta), 1-\epsilon, 1+\epsilon\big)\hat{A}_t \Big) \Big] + c_v \, \hat{\mathbb{E}}_t \Big[ \big(V_\theta(s_t) - \hat{R}_t\big)^2 \Big]$$
donde el ratio de verosimilitud en log-espacio se calcula como:
$$r_t(\theta) = \exp\Big( \log \pi_\theta(a_t \mid s_t) - \log \pi_{\theta_{\text{old}}}(a_t \mid s_t) \Big)$$


In [ ]:
# =====================================================================
# ARQUITECTURA ACTOR-CRÍTICO CONTINUA Y FUNCIÓN DE PÉRDIDA PPO
# =====================================================================


class ContinuousActorCritic(nn.Module):
    """Política Gaussiana diagonal pi_theta(a|s) y Crítico V(s) para HalfCheetah-v4."""

    def __init__(self, obs_dim: int = 17, act_dim: int = 6, hidden: int = 32):
        super().__init__()
        self.actor_backbone = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, act_dim),
        )
        for m in self.actor_backbone.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=0.05)
                nn.init.zeros_(m.bias)

        self.action_bias = nn.Parameter(torch.zeros(act_dim))
        self.log_std = nn.Parameter(torch.full((act_dim,), -0.8))

        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs: torch.Tensor):
        obs_norm = torch.clamp(obs / 5.0, -2.0, 2.0)
        mean = torch.tanh(0.25 * self.actor_backbone(obs_norm) + self.action_bias)
        std = torch.exp(self.log_std.clamp(-1.8, -0.3))
        return Normal(mean, std), self.critic(obs_norm).squeeze(-1)


def collect_rollouts(
    env: HalfCheetahVelEnv,
    policy: ContinuousActorCritic,
    T: int = 50,
    gamma: float = 0.50,
    lam: float = 0.80,
    seed: int = 0,
    deterministic: bool = False,
):
    """Recolecta un rollout de T pasos en la tarea tau y calcula ventajas GAE."""
    obs, _ = env.reset(seed=seed)
    S, A, LP, V, R, D, VELS, ERRS = [], [], [], [], [], [], [], []

    for t in range(T):
        s_t = torch.FloatTensor(obs).unsqueeze(0)
        with torch.no_grad():
            dist, val = policy(s_t)
            if deterministic:
                a_t = dist.mean
            else:
                if t % 5 == 0:
                    z = 0.75 * torch.randn(1, 1) + 0.25 * torch.randn_like(dist.mean)
                a_t = dist.mean + dist.stddev * z
            lp_t = dist.log_prob(a_t).mean(dim=-1)

        obs_next, r, term, trunc, info = env.step(a_t.squeeze(0).numpy())
        done = term or trunc

        S.append(s_t.squeeze(0))
        A.append(a_t.squeeze(0))
        LP.append(lp_t.squeeze(0))
        V.append(val.squeeze(0))
        R.append(float(r))
        D.append(float(done))
        VELS.append(float(info["x_velocity"]))
        ERRS.append(float(info["vel_error"]))

        obs = obs_next
        if done and t < T - 1:
            obs, _ = env.reset(seed=seed + t + 1)

    V_t = torch.stack(V)
    with torch.no_grad():
        nv = policy(torch.FloatTensor(obs).unsqueeze(0))[1].item()

    adv = torch.zeros(T)
    gae = 0.0
    for t in reversed(range(T)):
        mask = 1.0 - D[t]
        vnxt = V_t[t + 1].item() if t < T - 1 else nv
        delta = R[t] + gamma * vnxt * mask - V_t[t].item()
        gae = delta + gamma * lam * mask * gae
        adv[t] = gae

    steady_vel = float(np.mean(VELS[5:]))
    steady_err = abs(steady_vel - env.target_vel)

    return {
        "obs": torch.stack(S),
        "acts": torch.stack(A),
        "log_ps": torch.stack(LP),
        "vals": V_t,
        "advs": adv,
        "rets": adv + V_t,
        "ep_return": float(np.sum(R)),
        "mean_vel": steady_vel,
        "mean_err": steady_err,
    }


def ppo_loss(
    policy: ContinuousActorCritic,
    rollouts: dict,
    clip_eps: float = 0.25,
    c_v: float = 0.2,
):
    """Calcula la pérdida combinada Actor (L_CLIP) + Crítico (MSE)."""
    adv = rollouts["advs"].detach()
    adv_n = (adv - adv.mean()) / (adv.std() + 1e-8)

    dist, vals = policy(rollouts["obs"])
    new_lp = dist.log_prob(rollouts["acts"]).mean(dim=-1)
    old_lp = rollouts["log_ps"].detach()

    # TODO 2: Calcula el ratio = exp(new_lp - old_lp), los términos surr1 y surr2 recortado en [1-clip_eps, 1+clip_eps],
    # actor_loss = -min(surr1, surr2).mean() y critic_loss = MSE(vals, rollouts["rets"].detach()).
    # Retorna actor_loss + c_v * critic_loss
    # === TU CÓDIGO AQUÍ ===
    pass

---

## 4. Algoritmo Central: `MAML-PPO` con _Support Rollouts_ y _Query Rollouts_

Tal como revisamos en las **Diapositivas 16, 17, 19 y 22** de clase, cada paso de meta-entrenamiento de `MAML-PPO` sobre un lote (_batch_) de tareas $\{\tau_1, \dots, \tau_B\}$ ejecuta dos bucles anidados:

1. **Inner Loop (Adaptación rápida en el _Support Set_):**
   - Para cada tarea $\tau_i$, copiamos la política meta actual: $\theta_{\text{adapted}} \leftarrow \text{deepcopy}(\theta)$.
   - Recolectamos un rollout de soporte (`support_rollouts`) interactuando con $\tau_i$ y actualizamos $\theta_{\text{adapted}}$ mediante descenso de gradiente con tasa $\alpha$:
     $$\theta'_i = \theta - \alpha \nabla_\theta \mathcal{L}_{\text{PPO}}\big(\theta, \mathcal{D}_{\tau_i}^{\text{support}}\big)$$
2. **Outer Loop (Meta-Optimización en el _Query Set_ usando FOMAML):**
   - Recolectamos un **nuevo rollout de validación (`val_rollouts` / _Query Set_)** en la misma tarea $\tau_i$ utilizando la política ya adaptada $\theta'_i$.
   - Calculamos el gradiente de la pérdida PPO sobre `val_rollouts` ($g_{\text{FOMAML}} = \nabla_{\theta'_i} \mathcal{L}_{\text{PPO}}(\theta'_i, \mathcal{D}_{\tau_i}^{\text{query}})$, ignorando el Hessiano de segundo orden $\nabla_\theta^2 \mathcal{L}$ para evitar explosión de memoria) y acumulamos dicho meta-gradiente sobre los parámetros originales $\theta$ para actualizarlos con el meta-optimizador `Adam(lr=β)`.


In [ ]:
# =====================================================================
# IMPLEMENTACIÓN DE MAML-PPO (INNER LOOP + OUTER LOOP EN QUERY SET)
# =====================================================================


def maml_ppo_step(
    policy: ContinuousActorCritic,
    meta_opt: optim.Optimizer,
    task_envs: list,
    alpha: float = 0.45,
    n_inner: int = 1,
    T: int = 50,
    seed: int = 0,
):
    """
    Ejecuta una meta-iteración de MAML-PPO (FOMAML) sobre un lote de tareas `task_envs`.
    """
    meta_opt.zero_grad()
    grad_accum = [torch.zeros_like(p.data) for p in policy.parameters()]
    query_returns, query_errors = [], []

    for i, tau_env in enumerate(task_envs):
        # ── PASO 1: Inner Loop (Adaptación sobre Support Set) ──
        theta_adapted = copy.deepcopy(policy)
        inner_opt = optim.SGD(theta_adapted.parameters(), lr=alpha)

        for k in range(n_inner):
            # TODO 3.A: Recolecta `support_rollouts` en `tau_env` usando `theta_adapted`
            # y realiza 3 pasos internos de actualización sobre `theta_adapted` minimizando `ppo_loss`
            # === TU CÓDIGO AQUÍ ===
            pass

        # ── PASO 2: Outer Loop (Evaluación sobre Query Set nuevo con theta_adapted) ──
        # TODO 3.B: Recolecta un nuevo rollout de consulta `val_rollouts` en `tau_env` usando `theta_adapted`,
        # calcula `val_loss = ppo_loss(theta_adapted, val_rollouts)` y ejecuta `val_loss.backward()`
        # === TU CÓDIGO AQUÍ ===
        val_rollouts = None

        # Acumular el meta-gradiente de primer orden (FOMAML) hacia los parámetros de origen theta
        for g_acc, p_adap in zip(grad_accum, theta_adapted.parameters()):
            if p_adap.grad is not None:
                g_acc.add_(p_adap.grad.data / len(task_envs))

        query_returns.append(val_rollouts["ep_return"])
        query_errors.append(val_rollouts["mean_err"])

    # Aplicar el meta-gradiente promedio sobre la política origen `policy`
    for p, g_acc in zip(policy.parameters(), grad_accum):
        p.grad = g_acc.clone()
    meta_opt.step()

    return float(np.mean(query_returns)), float(np.mean(query_errors))


# ── Preparar políticas iniciales y entrenar MAML-PPO ──
np.random.seed(SEED)
torch.manual_seed(SEED)

# Política inicial Scratch (desajustada a velocidad baja y con exploración sub-óptima)
init_policy = ContinuousActorCritic()
with torch.no_grad():
    init_policy.action_bias.fill_(-0.85)
    init_policy.log_std.fill_(-1.45)

# Política PPO Especializada en una única tarea extrema (v = 2.25 m/s, baja entropía)
specialized_policy = ContinuousActorCritic()
with torch.no_grad():
    specialized_policy.action_bias.fill_(1.25)
    specialized_policy.log_std.fill_(-1.75)

maml_policy = copy.deepcopy(init_policy)
with torch.no_grad():
    maml_policy.log_std.fill_(-0.75)
meta_opt = optim.Adam(
    [maml_policy.action_bias]
    + list(maml_policy.actor_backbone.parameters())
    + list(maml_policy.critic.parameters()),
    lr=0.07,
)
maml_returns_hist, maml_errors_hist, maml_center_vels = [], [], []
env_probe = task_dist.sample(1.30)

print(
    "=== Iniciando Meta-Entrenamiento MAML-PPO (30 meta-iteraciones, Batch=4 tareas) ==="
)
for it in range(30):
    vels = [
        np.random.uniform(0.35, 0.80),
        np.random.uniform(0.80, 1.30),
        np.random.uniform(1.30, 1.80),
        np.random.uniform(1.80, 2.25),
    ]
    batch_envs = [task_dist.sample(v) for v in vels]
    qret, qerr = maml_ppo_step(
        maml_policy, meta_opt, batch_envs, alpha=0.48, n_inner=1, T=50, seed=it * 50
    )
    probe_ro = collect_rollouts(
        env_probe, maml_policy, T=50, seed=999, deterministic=True
    )

    maml_returns_hist.append(qret)
    maml_errors_hist.append(qerr)
    maml_center_vels.append(probe_ro["mean_vel"])

    if (it + 1) % 5 == 0 or it == 0:
        print(
            f"Iteración {it+1:2d}/30 | Query Return (1-shot): {qret:6.2f} | Error vel post-adaptación: {qerr:.3f} m/s | Vel centro theta: {probe_ro['mean_vel']:.2f} m/s"
        )

---

## 5. Alternativa Escalable de Primer Orden: `Reptile-PPO`

Como vimos en la **Diapositiva 20**, **Reptile (Nichol et al., 2018)** simplifica el meta-aprendizaje eliminando la necesidad de recolectar un _Query Set_ separado en el _Outer Loop_.  
Tras adaptar la política en cada tarea $\tau_i$ durante el _Inner Loop_ hasta obtener los pesos $\tilde{\theta}_i$, Reptile actualiza la inicialización $\theta$ mediante una **interpolación lineal de pesos** en dirección al promedio de los parámetros adaptados:
$$\theta \leftarrow \theta + \varepsilon \, \frac{1}{B} \sum_{i=1}^{B} \big( \tilde{\theta}_i - \theta \big)$$


In [ ]:
# =====================================================================
# IMPLEMENTACIÓN DE REPTILE-PPO (INTERPOLACIÓN LINEAL DE PESOS)
# =====================================================================


def reptile_ppo_step(
    policy: ContinuousActorCritic,
    task_envs: list,
    alpha: float = 0.45,
    eps_reptile: float = 0.28,
    n_inner: int = 1,
    T: int = 50,
    seed: int = 0,
):
    """
    Ejecuta una meta-iteración de Reptile-PPO sobre un lote de tareas `task_envs`.
    """
    orig_params = [p.data.clone() for p in policy.parameters()]
    delta_accum = [torch.zeros_like(p.data) for p in policy.parameters()]
    inner_returns, inner_errors = [], []

    for i, tau_env in enumerate(task_envs):
        theta_tilde = copy.deepcopy(policy)
        inner_opt = optim.SGD(theta_tilde.parameters(), lr=alpha)

        for k in range(n_inner):
            rollouts = collect_rollouts(
                tau_env, theta_tilde, T=T, seed=seed + 10 * i + k
            )
            for _ in range(3):
                loss = ppo_loss(theta_tilde, rollouts)
                inner_opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(theta_tilde.parameters(), 1.0)
                inner_opt.step()

        eval_ro = collect_rollouts(
            tau_env, theta_tilde, T=T, seed=seed + 200 + 10 * i, deterministic=True
        )
        inner_returns.append(eval_ro["ep_return"])
        inner_errors.append(eval_ro["mean_err"])

        # TODO 4.A: Acumula la diferencia de pesos (p_tilde.data - p_orig) / len(task_envs) en delta_accum
        # === TU CÓDIGO AQUÍ ===
        pass

    # TODO 4.B: Actualiza in-place los parámetros de `policy` sumando eps_reptile * d_acc
    # === TU CÓDIGO AQUÍ ===
    pass

    return float(np.mean(inner_returns)), float(np.mean(inner_errors))


# ── Entrenar Reptile-PPO desde la misma inicialización para comparación justa ──
np.random.seed(SEED)
torch.manual_seed(SEED)
reptile_policy = copy.deepcopy(init_policy)
with torch.no_grad():
    reptile_policy.log_std.fill_(-0.82)
reptile_returns_hist, reptile_errors_hist, reptile_center_vels = [], [], []

print(
    "=== Iniciando Meta-Entrenamiento Reptile-PPO (30 meta-iteraciones, Batch=4 tareas) ==="
)
for it in range(30):
    vels = [
        np.random.uniform(0.35, 0.80),
        np.random.uniform(0.80, 1.30),
        np.random.uniform(1.30, 1.80),
        np.random.uniform(1.80, 2.25),
    ]
    batch_envs = [task_dist.sample(v) for v in vels]
    iret, ierr = reptile_ppo_step(
        reptile_policy,
        batch_envs,
        alpha=0.45,
        eps_reptile=0.35,
        n_inner=1,
        T=50,
        seed=it * 50,
    )
    with torch.no_grad():
        reptile_policy.log_std.fill_(-0.82)
    probe_ro = collect_rollouts(
        env_probe, reptile_policy, T=50, seed=999, deterministic=True
    )

    reptile_returns_hist.append(iret)
    reptile_errors_hist.append(ierr)
    reptile_center_vels.append(probe_ro["mean_vel"])

    if (it + 1) % 5 == 0 or it == 0:
        print(
            f"Iteración {it+1:2d}/30 | Inner Return (1-shot): {iret:6.2f} | Error vel post-adaptación: {ierr:.3f} m/s | Vel centro theta: {probe_ro['mean_vel']:.2f} m/s"
        )

In [ ]:
# =====================================================================
# VISUALIZACIÓN 1: CURVA DE META-ENTRENAMIENTO Y CENTRO DE GRAVEDAD
# =====================================================================
fig1, axes1 = plt.subplots(1, 2, figsize=(13, 4.8))
iters = np.arange(1, 31)

axes1[0].plot(
    iters,
    maml_returns_hist,
    "o-",
    color="#1f77b4",
    lw=2.2,
    ms=4,
    label="MAML-PPO (Query Set 1-shot)",
)
axes1[0].plot(
    iters,
    reptile_returns_hist,
    "s--",
    color="#ff7f0e",
    lw=2.0,
    ms=4,
    label="Reptile-PPO (Inner 1-shot)",
)
axes1[0].set_title(
    "Curva de Meta-Entrenamiento (Retorno 1-Shot)", fontsize=12, fontweight="bold"
)
axes1[0].set_xlabel("Meta-Iteración (Outer Loop)")
axes1[0].set_ylabel("Retorno Episódico Post-Adaptación")
axes1[0].grid(alpha=0.3)
axes1[0].legend()

axes1[1].plot(
    iters,
    maml_center_vels,
    "o-",
    color="#1f77b4",
    lw=2.2,
    ms=4,
    label="Velocidad Inicial θ (MAML-PPO)",
)
axes1[1].plot(
    iters,
    reptile_center_vels,
    "s--",
    color="#ff7f0e",
    lw=2.0,
    ms=4,
    label="Velocidad Inicial θ (Reptile-PPO)",
)
axes1[1].axhline(
    1.30, color="crimson", ls=":", lw=2.0, label="Centro de Gravedad E[v_τ] = 1.30 m/s"
)
axes1[1].fill_between(
    iters, 0.35, 2.25, color="gray", alpha=0.10, label="Soporte p(τ) ∈ [0.35, 2.25] m/s"
)
axes1[1].set_title(
    "Interpretación Geométrica: Convergencia de θ al Centro de Gravedad",
    fontsize=12,
    fontweight="bold",
)
axes1[1].set_xlabel("Meta-Iteración (Outer Loop)")
axes1[1].set_ylabel("Velocidad Zero-Shot v_x(θ) [m/s]")
axes1[1].set_ylim(0.2, 2.4)
axes1[1].grid(alpha=0.3)
axes1[1].legend(loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

---

## 6. Evaluación _Meta-Test_ ($K$-Shot Adaptation) en Tareas Nuevas

Durante la fase de **Meta-Prueba (_Meta-Test_, Diapositivas 12, 13 y 23)**, congelamos el bucle externo y evaluamos qué tan rápido se adapta cada punto de inicialización $\theta$ ante dos tareas de locomoción opuestas:

1. **Tarea Nueva A (Trote lento controlado):** $v_{\text{test}} = 0.45\text{ m/s}$.
2. **Tarea Nueva B (Sprint de alta velocidad):** $v_{\text{test}} = 2.15\text{ m/s}$.

Para cada método (**MAML-PPO**, **Reptile-PPO**, **PPO Especializado en $v=2.25\text{ m/s}$** y **Política Inicial sin Meta-Entrenamiento**), mediremos la velocidad alcanzada y el error $|v_x - v_{\text{test}}|$ tras $K \in \{0, 1, 2, 3, 4, 5\}$ pasos de adaptación en el _Inner Loop_.


In [ ]:
# =====================================================================
# PROTOCOLO DE EVALUACIÓN META-TEST (ADAPTACIÓN K-SHOT)
# =====================================================================


def meta_test_adaptation(
    policy: ContinuousActorCritic,
    env_test: HalfCheetahVelEnv,
    K_steps: int = 5,
    alpha: float = 0.45,
    T: int = 50,
    seed: int = 500,
):
    """
    Evalúa la adaptación K-shot partiendo de `policy` en una nueva tarea `env_test` durante K_steps pasos.
    """
    adapted = copy.deepcopy(policy)
    inner_opt = optim.SGD(adapted.parameters(), lr=alpha)

    returns_k, errors_k, vels_k = [], [], []
    for k in range(K_steps + 1):
        # Evaluar desempeño actual de la política (en k=0 evalúa Zero-Shot antes de adaptar)
        eval_ro = collect_rollouts(
            env_test, adapted, T=T, seed=seed + k, deterministic=True
        )
        returns_k.append(eval_ro["ep_return"])
        errors_k.append(eval_ro["mean_err"])
        vels_k.append(eval_ro["mean_vel"])

        if k < K_steps:
            # TODO 5: Recolecta 1 Support Rollout estocástico (`deterministic=False`) en `env_test`
            # y ejecuta 3 actualizaciones de gradiente con `inner_opt` minimizando `ppo_loss(adapted, train_ro)`
            # === TU CÓDIGO AQUÍ ===
            pass

    return {
        "returns": np.array(returns_k),
        "errors": np.array(errors_k),
        "vels": np.array(vels_k),
    }


env_slow = task_dist.sample(0.45)
env_fast = task_dist.sample(2.15)

res_slow = {
    "MAML-PPO": meta_test_adaptation(maml_policy, env_slow, K_steps=5, alpha=0.48),
    "Reptile-PPO": meta_test_adaptation(
        reptile_policy, env_slow, K_steps=5, alpha=0.45
    ),
    "PPO Especializado (v=2.25)": meta_test_adaptation(
        specialized_policy, env_slow, K_steps=5, alpha=0.45
    ),
    "Scratch (Sin Meta-Train)": meta_test_adaptation(
        init_policy, env_slow, K_steps=5, alpha=0.45
    ),
}
res_fast = {
    "MAML-PPO": meta_test_adaptation(maml_policy, env_fast, K_steps=5, alpha=0.48),
    "Reptile-PPO": meta_test_adaptation(
        reptile_policy, env_fast, K_steps=5, alpha=0.45
    ),
    "PPO Especializado (v=2.25)": meta_test_adaptation(
        specialized_policy, env_fast, K_steps=5, alpha=0.45
    ),
    "Scratch (Sin Meta-Train)": meta_test_adaptation(
        init_policy, env_fast, K_steps=5, alpha=0.45
    ),
}

for title, v_t, res_dict in [
    ("TAREA NUEVA A (Trote Lento)", 0.45, res_slow),
    ("TAREA NUEVA B (Sprint Rápido)", 2.15, res_fast),
]:
    print(f"=== {title}: Velocidad Objetivo v_tau = {v_t:.2f} m/s ===")
    print(
        "Método                       |  K=0 (0-shot) |  K=1 (1-shot) |  K=3 (3-shot) |  K=5 (5-shot) | Error final (m/s)"
    )
    print("-" * 108)
    for name, d in res_dict.items():
        v0, v1, v3, v5 = d["vels"][0], d["vels"][1], d["vels"][3], d["vels"][5]
        e5 = d["errors"][5]
        print(
            f"{name:28s} |  v={v0:5.2f} m/s  |  v={v1:5.2f} m/s  |  v={v3:5.2f} m/s  |  v={v5:5.2f} m/s  |   {e5:5.3f} m/s"
        )
    print()

In [ ]:
# =====================================================================
# VISUALIZACIÓN 2: COMPARACIÓN K-SHOT EN TAREAS NUEVAS DE META-TEST
# =====================================================================
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5.0))
ks = np.arange(0, 6)
colors = {
    "MAML-PPO": "#1f77b4",
    "Reptile-PPO": "#ff7f0e",
    "PPO Especializado (v=2.25)": "#d62728",
    "Scratch (Sin Meta-Train)": "#7f7f7f",
}
markers = {
    "MAML-PPO": "o-",
    "Reptile-PPO": "s--",
    "PPO Especializado (v=2.25)": "^-.",
    "Scratch (Sin Meta-Train)": "x:",
}

for name, d in res_slow.items():
    axes2[0].plot(
        ks, d["vels"], markers[name], color=colors[name], lw=2.2, ms=6, label=name
    )
axes2[0].axhline(0.45, color="black", ls="--", lw=1.8, label="Meta v_τ = 0.45 m/s")
axes2[0].set_title(
    "Meta-Test A: Adaptación a v_τ = 0.45 m/s", fontsize=11, fontweight="bold"
)
axes2[0].set_xlabel("Pasos de Gradiente en Inner Loop (K-shots)")
axes2[0].set_ylabel("Velocidad Alcanzada v_x [m/s]")
axes2[0].set_ylim(0.2, 2.35)
axes2[0].grid(alpha=0.3)
axes2[0].legend(fontsize=8.5)

for name, d in res_fast.items():
    axes2[1].plot(
        ks, d["vels"], markers[name], color=colors[name], lw=2.2, ms=6, label=name
    )
axes2[1].axhline(2.15, color="black", ls="--", lw=1.8, label="Meta v_τ = 2.15 m/s")
axes2[1].set_title(
    "Meta-Test B: Adaptación a v_τ = 2.15 m/s", fontsize=11, fontweight="bold"
)
axes2[1].set_xlabel("Pasos de Gradiente en Inner Loop (K-shots)")
axes2[1].set_ylabel("Velocidad Alcanzada v_x [m/s]")
axes2[1].set_ylim(0.2, 2.35)
axes2[1].grid(alpha=0.3)
axes2[1].legend(fontsize=8.5)

for name in res_slow.keys():
    mean_err_both = 0.5 * (res_slow[name]["errors"] + res_fast[name]["errors"])
    axes2[2].plot(
        ks, mean_err_both, markers[name], color=colors[name], lw=2.2, ms=6, label=name
    )
axes2[2].set_title(
    "Error Medio |v_x − v_τ| Promedio en Meta-Test", fontsize=11, fontweight="bold"
)
axes2[2].set_xlabel("Pasos de Gradiente en Inner Loop (K-shots)")
axes2[2].set_ylabel("Error Absoluto de Velocidad [m/s]")
axes2[2].grid(alpha=0.3)
axes2[2].legend(fontsize=8.5)

plt.suptitle(
    "Meta-RL en MuJoCo HalfCheetah-v4: Adaptación Few-Shot (MAML-PPO vs. Reptile-PPO vs. Baselines)",
    fontsize=13,
    fontweight="bold",
    y=1.03,
)
plt.tight_layout()
plt.show()

---
## 7. Preguntas de Análisis Crítico

Responde de manera fundamentada y haciendo referencia a las métricas y gráficas obtenidas en tus experimentos:

### Pregunta 7.1: Rol del *Support Set* vs. *Query Set* en `MAML-PPO`
¿Por qué en el *Outer Loop* de `MAML-PPO` es indispensable recolectar un **nuevo rollout de validación (`val_rollouts` / *Query Set*)** ejecutando la política ya adaptada $\theta'_\tau$, en lugar de actualizar la política meta $\theta$ directamente con los gradientes del primer rollout (`support_rollouts`)? ¿Qué le ocurriría a la inicialización $\theta$ si no evaluáramos sobre el desempeño post-adaptación?

**[RESPUESTA DEL ESTUDIANTE]:**
> *Escribe tu respuesta aquí...*

---

### Pregunta 7.2: Interpretación Geométrica (_"Centro de Gravedad"_) vs. Especialización Excesiva

Observando la **Figura 1 (panel derecho)** y la **Figura 2**:

1. ¿Hacia qué velocidad zero-shot ($K=0$) convergieron los parámetros meta $\theta^*$ de `MAML-PPO` y `Reptile-PPO`, y cómo se relaciona este valor con la esperanza de la distribución de tareas $\mathbb{E}[v_\tau]$ para $v_\tau \sim \mathcal{U}[0.35, 2.25]\text{ m/s}$?
2. ¿Por qué el agente **`PPO Especializado (v=2.25)`** falla drásticamente al intentar adaptarse a la Tarea A ($v_\tau = 0.45\text{ m/s}$) en $K=5$ pasos, pese a ser experto en correr a alta velocidad?

**[RESPUESTA DEL ESTUDIANTE]:**

> _Escribe tu respuesta aquí..._

---

### Pregunta 7.3: Compromiso Computacional y Empírico entre `MAML-PPO` (FOMAML) y `Reptile-PPO`

Compara cuantitativamente cuántos _rollouts_ de interacción con MuJoCo ejecuta por cada meta-iteración `MAML-PPO` frente a `Reptile-PPO` (con tamaño de lote $B=4$ y $n_{\text{inner}}=1$). Considerando además el costo del Hessiano $\nabla_\theta^2 \mathcal{L}$ en MAML de segundo orden original, ¿en qué escenarios de robótica real preferirías usar FOMAML/MAML-PPO y en cuáles Reptile?

**[RESPUESTA DEL ESTUDIANTE]:**

> _Escribe tu respuesta aquí..._
